# 01 - Dataset Understanding

## AI-Assisted Hospital Operations & Patient Analytics Dashboard

This notebook starts Stage 2 of the project: **Dataset Selection & Understanding**.

The goal of this notebook is to understand the raw HMIS dataset before cleaning, SQL analysis, or dashboard building.

In this notebook, we will:

- confirm available CSV files
- load each dataset using pandas
- check rows, columns, and data types
- check missing values and duplicate rows
- identify primary keys and foreign keys
- document early healthcare analytics observations


## 1. Import Libraries

We start with basic Python libraries used for data analysis.

- `pandas` helps us load and inspect tables.
- `Path` helps us work with folder paths in a clean way.


In [1]:
from pathlib import Path

import pandas as pd

# Show all columns when displaying dataframes
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Set Dataset Path

The raw CSV files are stored in `data/raw/hospital_data/`.

Because this notebook is inside the `notebooks/` folder, `..` means go one level up to the project root.

In [2]:
project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw" / "hospital_data"

raw_data_path

WindowsPath('C:/Users/devip/Desktop/Healthcare Projects/Healthcare-Analytics-Project/data/raw/hospital_data')

## 3. Confirm Expected Files

Before analysis, we confirm that all expected CSV files are available.

This is a simple but important analyst habit. If a file is missing, later joins and KPIs may fail.

In [3]:
expected_files = [
    "department.csv",
    "ward.csv",
    "bed.csv",
    "disease.csv",
    "diagnostic_test.csv",
    "drug_manufacturer.csv",
    "insurance_provider.csv",
    "patient.csv",
    "admission.csv",
    "billing.csv",
    "billing_detail.csv",
    "patient_diagnostic.csv",
    "drug.csv",
    "drug_inventory.csv",
    "prescription.csv",
    "patient_insurance.csv",
    "employee.csv",
    "doctor.csv",
    "staff_assignment.csv",
]

available_files = sorted([file.name for file in raw_data_path.glob("*.csv")])

file_check = pd.DataFrame({
    "expected_file": expected_files,
    "is_available": [file in available_files for file in expected_files]
})

file_check

,expected_file,is_available
0,department.csv,True
1,ward.csv,True
2,bed.csv,True
3,disease.csv,True
4,diagnostic_test.csv,True
5,drug_manufacturer.csv,True
6,insurance_provider.csv,True
7,patient.csv,True
8,admission.csv,True
9,billing.csv,True


In [4]:
missing_files = sorted(set(expected_files) - set(available_files))
extra_files = sorted(set(available_files) - set(expected_files))

print(f"Expected CSV files: {len(expected_files)}")
print(f"Available CSV files: {len(available_files)}")
print(f"Missing files: {missing_files}")
print(f"Extra files: {extra_files}")

Expected CSV files: 19
Available CSV files: 19
Missing files: []
Extra files: []


## 4. Load All CSV Files

We will load each CSV into a pandas DataFrame.

The dictionary `tables` stores each table using a simple table name. For example:

- `tables["patient"]` gives the patient table
- `tables["admission"]` gives the admission table
- `tables["billing"]` gives the billing table

In [5]:
tables = {}

for file_name in expected_files:
    table_name = file_name.replace(".csv", "")
    file_path = raw_data_path / file_name
    tables[table_name] = pd.read_csv(file_path)

list(tables.keys())

['department',
 'ward',
 'bed',
 'disease',
 'diagnostic_test',
 'drug_manufacturer',
 'insurance_provider',
 'patient',
 'admission',
 'billing',
 'billing_detail',
 'patient_diagnostic',
 'drug',
 'drug_inventory',
 'prescription',
 'patient_insurance',
 'employee',
 'doctor',
 'staff_assignment']

## 5. Create Dataset Summary

This summary gives a quick overview of each table:

- number of rows
- number of columns
- duplicate rows
- missing values
- column names


In [6]:
summary_rows = []

for table_name, df in tables.items():
    summary_rows.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values_total": int(df.isna().sum().sum()),
        "column_names": ", ".join(df.columns)
    })

dataset_summary = pd.DataFrame(summary_rows).sort_values("table_name")
dataset_summary

,table_name,rows,columns,duplicate_rows,missing_values_total,column_names
8,admission,45000,10,0,0,"admission_id, admission_date, discharge_date, admission_type, admission_status, patient_id, department_id, ward_id, ..."
2,bed,415,4,0,0,"bed_id, bed_number, bed_status, ward_id"
9,billing,45000,8,0,0,"bill_id, bill_date, total_amount, insurance_covered_amount, patient_payable_amount, payment_status, payment_mode, ad..."
10,billing_detail,112402,5,0,67402,"billing_detail_id, charge_type, reference_id, amount, bill_id"
0,department,11,5,0,0,"department_id, department_name, department_type, floor_number, status"
4,diagnostic_test,9,5,0,0,"test_id, test_name, test_category, standard_cost, department_id"
3,disease,20,3,0,0,"disease_id, disease_name, disease_category"
17,doctor,98,5,0,0,"doctor_id, employee_id, specialization, qualification, experience_years"
12,drug,250,6,0,0,"drug_id, drug_name, brand_name, drug_category, unit_cost, manufacturer_id"
13,drug_inventory,250,6,0,0,"inventory_id, current_stock, reorder_level, inventory_status, last_restock_date, drug_id"


## 6. Check Data Types

Data types tell us how pandas understood each column.

Important note: date columns often load as `object` first. In Stage 3, we will convert date columns into proper datetime format.

In [7]:
dtype_rows = []

for table_name, df in tables.items():
    for column in df.columns:
        dtype_rows.append({
            "table_name": table_name,
            "column_name": column,
            "data_type": str(df[column].dtype),
            "missing_values": int(df[column].isna().sum()),
            "unique_values": int(df[column].nunique(dropna=True))
        })

data_types_summary = pd.DataFrame(dtype_rows)
data_types_summary.head(20)

,table_name,column_name,data_type,missing_values,unique_values
0,department,department_id,int64,0,11
1,department,department_name,object,0,11
2,department,department_type,object,0,3
3,department,floor_number,int64,0,5
4,department,status,object,0,1
5,ward,ward_id,int64,0,27
6,ward,ward_name,object,0,27
7,ward,ward_type,object,0,4
8,ward,total_beds,int64,0,3
9,ward,department_id,int64,0,6


## 7. Preview Important Tables

The central table is `admission` because most hospital analytics questions start from a patient admission or encounter.

In [8]:
tables["admission"].head()

,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id
0,1,2020-02-25,2020-02-27,Emergency,Discharged,166,2,6,76,10
1,2,2022-02-22,2022-03-04,Elective,Discharged,8622,5,21,302,11
2,3,2021-02-03,2021-02-09,Elective,Discharged,23976,1,2,11,9
3,4,2021-12-31,2022-01-05,Elective,Discharged,16635,2,10,128,1
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7


In [9]:
tables["patient"].head()

,patient_id,gender,date_of_birth,blood_group,city,contact_number
0,1,Female,1987-08-24,O-,East Stephanieberg,+1-792-342-0981
1,2,Male,1960-05-18,A-,Manuelbury,793-725-0800
2,3,Male,1955-04-24,A-,Lake Susanchester,+1-330-617-3749x232
3,4,Male,2004-06-16,B-,South Leslieburgh,426-611-6235x07684
4,5,Male,1977-07-22,A-,Lopezchester,(215)330-2831x0821


In [10]:
tables["billing"].head()

,bill_id,bill_date,total_amount,insurance_covered_amount,patient_payable_amount,payment_status,payment_mode,admission_id
0,1,2025-05-26,68483,61634.7,6848.3,Paid,Insurance,1
1,2,2023-11-19,70917,35458.5,35458.5,Paid,Insurance,2
2,3,2023-02-20,28137,25323.3,2813.7,Pending,Insurance,3
3,4,2021-09-01,80665,64532.0,16133.0,Pending,Insurance,4
4,5,2021-07-13,54920,27460.0,27460.0,Pending,Insurance,5


## 8. Missing Values by Table

Missing values are blank or unavailable values.

In healthcare analytics, missing values matter because they can affect KPIs such as average length of stay, billing totals, or patient demographic breakdowns.

In [11]:
missing_summary = []

for table_name, df in tables.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        if missing_count > 0:
            missing_summary.append({
                "table_name": table_name,
                "column_name": column,
                "missing_count": missing_count,
                "missing_percent": round((missing_count / len(df)) * 100, 2)
            })

missing_summary_df = pd.DataFrame(missing_summary)

if missing_summary_df.empty:
    print("No missing values found in the checked tables.")
else:
    display(missing_summary_df.sort_values(["table_name", "missing_percent"], ascending=[True, False]))

,table_name,column_name,missing_count,missing_percent
0,billing_detail,reference_id,67402,59.97


## 9. Duplicate Row Check

Duplicate rows can inflate counts, revenue, patient visits, and other KPIs.

At this stage, we only identify duplicates. We will decide cleaning actions in Stage 3.

In [12]:
duplicate_summary = []

for table_name, df in tables.items():
    duplicate_summary.append({
        "table_name": table_name,
        "duplicate_rows": int(df.duplicated().sum())
    })

pd.DataFrame(duplicate_summary).sort_values("table_name")

,table_name,duplicate_rows
8,admission,0
2,bed,0
9,billing,0
10,billing_detail,0
0,department,0
4,diagnostic_test,0
3,disease,0
17,doctor,0
12,drug,0
13,drug_inventory,0


## 10. Primary Key Uniqueness Check

A primary key should uniquely identify each row in a table.

For example, every `patient_id` should represent one patient record in `patient.csv`.

In [13]:
primary_keys = {
    "department": "department_id",
    "ward": "ward_id",
    "bed": "bed_id",
    "disease": "disease_id",
    "diagnostic_test": "test_id",
    "drug_manufacturer": "manufacturer_id",
    "insurance_provider": "insurance_provider_id",
    "patient": "patient_id",
    "admission": "admission_id",
    "billing": "bill_id",
    "billing_detail": "billing_detail_id",
    "patient_diagnostic": "patient_diagnostic_id",
    "drug": "drug_id",
    "drug_inventory": "inventory_id",
    "prescription": "prescription_id",
    "patient_insurance": "patient_insurance_id",
    "employee": "employee_id",
    "doctor": "doctor_id",
    "staff_assignment": "assignment_id",
}

pk_summary = []

for table_name, pk_column in primary_keys.items():
    df = tables[table_name]
    pk_summary.append({
        "table_name": table_name,
        "primary_key": pk_column,
        "rows": len(df),
        "unique_primary_keys": df[pk_column].nunique(dropna=True),
        "duplicate_primary_keys": int(df[pk_column].duplicated().sum()),
        "missing_primary_keys": int(df[pk_column].isna().sum())
    })

pd.DataFrame(pk_summary).sort_values("table_name")

,table_name,primary_key,rows,unique_primary_keys,duplicate_primary_keys,missing_primary_keys
8,admission,admission_id,45000,45000,0,0
2,bed,bed_id,415,415,0,0
9,billing,bill_id,45000,45000,0,0
10,billing_detail,billing_detail_id,112402,112402,0,0
0,department,department_id,11,11,0,0
4,diagnostic_test,test_id,9,9,0,0
3,disease,disease_id,20,20,0,0
17,doctor,doctor_id,98,98,0,0
12,drug,drug_id,250,250,0,0
13,drug_inventory,inventory_id,250,250,0,0


## 11. Basic Relationship Checks

Relationship checks confirm whether foreign keys in one table match valid primary keys in another table.

Example: every `patient_id` in `admission.csv` should exist in `patient.csv`.

In [14]:
relationship_checks = [
    ("admission", "patient_id", "patient", "patient_id"),
    ("admission", "department_id", "department", "department_id"),
    ("admission", "ward_id", "ward", "ward_id"),
    ("admission", "bed_id", "bed", "bed_id"),
    ("admission", "disease_id", "disease", "disease_id"),
    ("billing", "admission_id", "admission", "admission_id"),
    ("billing_detail", "bill_id", "billing", "bill_id"),
    ("patient_diagnostic", "admission_id", "admission", "admission_id"),
    ("patient_diagnostic", "test_id", "diagnostic_test", "test_id"),
    ("patient_diagnostic", "doctor_id", "doctor", "doctor_id"),
    ("prescription", "admission_id", "admission", "admission_id"),
    ("prescription", "drug_id", "drug", "drug_id"),
    ("patient_insurance", "patient_id", "patient", "patient_id"),
    ("patient_insurance", "insurance_provider_id", "insurance_provider", "insurance_provider_id"),
    ("doctor", "employee_id", "employee", "employee_id"),
    ("ward", "department_id", "department", "department_id"),
    ("bed", "ward_id", "ward", "ward_id"),
    ("drug", "manufacturer_id", "drug_manufacturer", "manufacturer_id"),
    ("drug_inventory", "drug_id", "drug", "drug_id"),
    ("staff_assignment", "employee_id", "employee", "employee_id"),
    ("staff_assignment", "ward_id", "ward", "ward_id"),
]

relationship_results = []

for child_table, child_key, parent_table, parent_key in relationship_checks:
    child_values = set(tables[child_table][child_key].dropna().unique())
    parent_values = set(tables[parent_table][parent_key].dropna().unique())
    unmatched_values = child_values - parent_values
    relationship_results.append({
        "child_table": child_table,
        "child_key": child_key,
        "parent_table": parent_table,
        "parent_key": parent_key,
        "unmatched_key_count": len(unmatched_values)
    })

pd.DataFrame(relationship_results).sort_values("unmatched_key_count", ascending=False)

,child_table,child_key,parent_table,parent_key,unmatched_key_count
0,admission,patient_id,patient,patient_id,0
11,prescription,drug_id,drug,drug_id,0
19,staff_assignment,employee_id,employee,employee_id,0
18,drug_inventory,drug_id,drug,drug_id,0
17,drug,manufacturer_id,drug_manufacturer,manufacturer_id,0
16,bed,ward_id,ward,ward_id,0
15,ward,department_id,department,department_id,0
14,doctor,employee_id,employee,employee_id,0
13,patient_insurance,insurance_provider_id,insurance_provider,insurance_provider_id,0
12,patient_insurance,patient_id,patient,patient_id,0


## 12. Classify Tables by Analytical Area

Classifying tables helps us understand which tables support each dashboard.

In [15]:
table_classification = pd.DataFrame([
    {"table_name": "patient", "category": "Patient master", "main_use": "Demographics and patient profile analysis"},
    {"table_name": "admission", "category": "Central transaction", "main_use": "Admissions, discharges, length of stay, department workload"},
    {"table_name": "department", "category": "Hospital master", "main_use": "Department-level analysis"},
    {"table_name": "ward", "category": "Hospital master", "main_use": "Ward capacity and operations"},
    {"table_name": "bed", "category": "Hospital master", "main_use": "Bed status and utilization"},
    {"table_name": "disease", "category": "Clinical master", "main_use": "Disease trend and clinical grouping"},
    {"table_name": "diagnostic_test", "category": "Clinical master", "main_use": "Diagnostic test categories and costs"},
    {"table_name": "patient_diagnostic", "category": "Clinical transaction", "main_use": "Test usage and result patterns"},
    {"table_name": "prescription", "category": "Clinical transaction", "main_use": "Medication usage patterns"},
    {"table_name": "drug", "category": "Pharmacy master", "main_use": "Drug catalog and medication categories"},
    {"table_name": "drug_inventory", "category": "Pharmacy transaction", "main_use": "Stock level and reorder monitoring"},
    {"table_name": "drug_manufacturer", "category": "Pharmacy master", "main_use": "Manufacturer reliability and contract analysis"},
    {"table_name": "billing", "category": "Financial transaction", "main_use": "Revenue, insurance coverage, payment status"},
    {"table_name": "billing_detail", "category": "Financial transaction", "main_use": "Charge-level billing analysis"},
    {"table_name": "insurance_provider", "category": "Financial master", "main_use": "Insurance provider analysis"},
    {"table_name": "patient_insurance", "category": "Financial transaction", "main_use": "Patient insurance coverage"},
    {"table_name": "employee", "category": "Staff master", "main_use": "Staff count and department staffing"},
    {"table_name": "doctor", "category": "Staff master", "main_use": "Doctor specialization and experience"},
    {"table_name": "staff_assignment", "category": "Staff transaction", "main_use": "Ward staffing and shift assignments"},
])

table_classification

,table_name,category,main_use
0,patient,Patient master,Demographics and patient profile analysis
1,admission,Central transaction,"Admissions, discharges, length of stay, department workload"
2,department,Hospital master,Department-level analysis
3,ward,Hospital master,Ward capacity and operations
4,bed,Hospital master,Bed status and utilization
5,disease,Clinical master,Disease trend and clinical grouping
6,diagnostic_test,Clinical master,Diagnostic test categories and costs
7,patient_diagnostic,Clinical transaction,Test usage and result patterns
8,prescription,Clinical transaction,Medication usage patterns
9,drug,Pharmacy master,Drug catalog and medication categories


## 13. Early Dashboard Planning Notes

Based on the available tables, this dataset can support four dashboard pages.

### Executive Dashboard
- admissions trend
- total revenue
- patient volume
- department performance summary

### Clinical Dashboard
- disease categories
- test result status
- prescription patterns
- demographics by diagnosis

### Operations Dashboard
- admission type and status
- average length of stay
- department workload
- bed and ward capacity

### Financial Dashboard
- billing amount trends
- insurance covered vs patient payable amount
- payment status
- revenue by department and disease category


## 14. Stage 2 Findings and Next Steps

Use this section to write observations after running the notebook.

### Initial Findings
- The dataset contains 19 related CSV files.
- The admission table is the central transaction table.
- The dataset supports clinical, operational, financial, and staff analytics.
- Date columns will need conversion in Stage 3.

### Next Stage: Data Cleaning and Preprocessing
In Stage 3, we will:

- convert date columns to datetime format
- create age and age group fields
- calculate length of stay
- validate billing calculations
- prepare cleaned tables for SQL and Power BI
